In [6]:

import json
from litellm import completion
import sys
import os
from datetime import datetime
from typing import List, Optional, Dict, Any
import argparse
from dataclasses import dataclass


current_dir = os.getcwd()
sft_path = os.path.join(os.path.dirname(os.path.dirname(current_dir)), "sft_training")
if os.path.exists(sft_path) and sft_path not in sys.path:
    sys.path.insert(0, sft_path)

from tau_bench.agents.base import Agent
from tau_bench.agents import debug_message_to_action
from tau_bench.envs.base import Env
from tau_bench.types import SolveResult, Action, RESPOND_ACTION_NAME

####### cell #######

class ToolCallingAgent(Agent):
    def __init__(
        self,
        tools_info: Optional[List[Dict[str, Any]]],
        wiki: str,
        model: str,
        provider: str,
        temperature: float = 0.0,
    ):
        # DEBUGGING: Select specific tools for exchange scenario
        relevant_tool_names = [
            'find_user_id_by_name_zip', 
            'get_order_details', 
            'get_product_details',
            'exchange_delivered_order_items'
        ]
        
        # Handle None tools_info
        if tools_info is None:
            self.tools_info = []
        else:
            # self.tools_info = [tool for tool in tools_info if tool['function']['name'] in relevant_tool_names]
            self.tools_info = tools_info
        self.wiki = wiki
        self.model = model
        self.provider = provider
        self.temperature = temperature
        
        print(f"🔧 DEBUG: Agent initialized with:")
        print(f"   Model: {model}")
        print(f"   Provider: {provider}")
        print(f"   Temperature: {temperature}")
        print(f"   Wiki length: {len(wiki)}")
        print(f"   Tools count: {len(tools_info) if tools_info else 0} -> LIMITED TO: {len(self.tools_info)}")
        if self.tools_info:
            print(f"   Available tools: {[tool['function']['name'] for tool in self.tools_info]}")
        else:
            print(f"   Available tools: None (no tools provided)")
        
        # Handle custom providers by mapping them to litellm providers
        if provider == "gemma":
            # Auto-detect the appropriate litellm provider based on model name
            if model.startswith("gemma:") or ":" in model:
                self.litellm_provider = "ollama"
                self.litellm_model = model  # Keep the model as is for Ollama
                print(f"🔧 DEBUG: Mapped to Ollama - {model}")
            elif model.startswith("vllm/") or model.startswith("/"):
                # vLLM deployment (local path or vllm/ prefix)
                self.litellm_provider = "vllm"  
                self.litellm_model = model.replace("vllm/", "") if model.startswith("vllm/") else model
                print(f"🔧 DEBUG: Mapped to vLLM - {self.litellm_model}")
            else:
                # Assume HuggingFace format
                self.litellm_provider = "huggingface"
                self.litellm_model = f"google/{model}" if not model.startswith("google/") else model
                print(f"🔧 DEBUG: Mapped to HuggingFace - {self.litellm_model}")
        else:
            # Use provider and model as-is for standard litellm providers
            self.litellm_provider = provider
            self.litellm_model = model
            print(f"🔧 DEBUG: Using direct mapping - {provider}/{model}")

    def solve(
        self, env: Env, task_index: Optional[int] = None, max_num_steps: int = 30
    ) -> SolveResult:
        print(f"\n🚀 DEBUG: Starting solve for task {task_index}")
        
        # Create detailed log file
        log_filename = f"gemma3_debug_detailed_task_{task_index}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        
        def log_and_print(message):
            """Helper function to both print and log messages"""
            print(message)
            with open(log_filename, "a", encoding="utf-8") as log_file:
                log_file.write(message + "\n")
        
        log_and_print(f"🚀 DEBUG: Starting solve for task {task_index}")
        log_and_print(f"📅 Timestamp: {datetime.now().isoformat()}")
        log_and_print(f"🔧 Model: {self.model} via {self.provider}")
        log_and_print("=" * 100)
        
        total_cost = 0.0
        env_reset_res = env.reset(task_index=task_index)
        obs = env_reset_res.observation
        info = env_reset_res.info.model_dump()
        reward = 0.0
        
        log_and_print(f"🔧 DEBUG: Initial observation: {obs}")
        
        messages: List[Dict[str, Any]] = [
            {"role": "system", "content": self.wiki},
            {"role": "user", "content": obs},
        ]
        
        log_and_print(f"🔧 DEBUG: Initial messages constructed:")
        log_and_print(f"   System message length: {len(messages[0]['content'])}")
        log_and_print(f"   User message: {messages[1]['content']}")
        log_and_print(f"   🎯 Task index being run: {task_index}")
        
        # Print the full system message to understand what instructions are being sent
        log_and_print(f"\n📋 FULL SYSTEM MESSAGE (WIKI):")
        log_and_print("=" * 80)
        log_and_print(messages[0]['content'])
        log_and_print("=" * 80)
        
        log_and_print(f"\n📋 FULL USER INSTRUCTION:")
        log_and_print("=" * 80)
        log_and_print(messages[1]['content'])
        log_and_print("=" * 80)
        
        for step in range(max_num_steps):
            log_and_print(f"\n📍 DEBUG: turn {step + 1}/{max_num_steps}")
            log_and_print("🔥" * 60)
            
            # Show the exact parameters being sent
            log_and_print(f"🔧 DEBUG: LiteLLM call parameters:")
            log_and_print(f"   model: {self.litellm_model}")
            log_and_print(f"   custom_llm_provider: {self.litellm_provider}")
            log_and_print(f"   temperature: {self.temperature}")
            log_and_print(f"   tools count: {len(self.tools_info)}")
            log_and_print(f"   messages count: {len(messages)}")
            
            # Log the COMPLETE conversation context being sent to the model
            log_and_print(f"\n💬 COMPLETE CONVERSATION CONTEXT SENT TO MODEL:")
            log_and_print("🔸" * 80)
            for i, msg in enumerate(messages):
                log_and_print(f"MESSAGE {i+1} (Role: {msg['role']}):")
                log_and_print(f"Content: {msg.get('content', 'N/A')}")
                if 'tool_call_id' in msg:
                    log_and_print(f"Tool Call ID: {msg['tool_call_id']}")
                if 'name' in msg:
                    log_and_print(f"Tool Name: {msg['name']}")
                log_and_print("-" * 40)
            log_and_print("🔸" * 80)
            
            # Make the LiteLLM call - only pass tools if we have them
            completion_params = {
                "messages": messages,
                "model": self.litellm_model,
                "custom_llm_provider": self.litellm_provider,
                "temperature": self.temperature,
            }
            
            # Only add tools if we have them
            if self.tools_info:
                completion_params["tools"] = self.tools_info
                log_and_print(f"🔧 DEBUG: Including {len(self.tools_info)} tools in request")
            else:
                log_and_print(f"🔧 DEBUG: No tools included in request")
                
            try:
                res = completion(**completion_params)
                log_and_print(f"✅ DEBUG: LiteLLM response received")
                log_and_print(f"   Model: {res.model}")
                log_and_print(f"   Response cost: {res._hidden_params.get('response_cost', 0.0)}")
                
            except Exception as e:
                log_and_print(f"❌ DEBUG: LiteLLM call failed: {e}")
                break
            
            # Extract and analyze the message
            next_message = res.choices[0].message.model_dump()
            
            log_and_print(f"\n🤖 DEBUG: Raw message from model:")
            log_and_print(f"   Role: {next_message.get('role')}")
            log_and_print(f"   Content: {next_message.get('content')}")
            log_and_print(f"   Tool calls: {next_message.get('tool_calls')}")
            
            # Log the COMPLETE raw response
            log_and_print(f"\n📄 COMPLETE RAW MODEL RESPONSE:")
            log_and_print("🔹" * 80)
            log_and_print(json.dumps(next_message, indent=2, ensure_ascii=False))
            log_and_print("🔹" * 80)
            
            total_cost += res._hidden_params["response_cost"]
            
            # Convert to action with debugging
            action = debug_message_to_action(next_message)
            
            log_and_print(f"🎯 DEBUG: Action created:")
            log_and_print(f"   Name: {action.name}")
            log_and_print(f"   Kwargs: {action.kwargs}")
            
            # Execute the action
            env_response = env.step(action)
            reward = env_response.reward
            info = {**info, **env_response.info.model_dump()}
            
            log_and_print(f"🔧 DEBUG: Environment response:")
            log_and_print(f"   Reward: {reward}")
            log_and_print(f"   Done: {env_response.done}")
            log_and_print(f"   Observation: {env_response.observation}")
            
            # Handle conversation continuation
            if action.name != RESPOND_ACTION_NAME:
                log_and_print(f"🔧 DEBUG: Adding tool call to conversation")
                
                # Check if we have actual tool_calls or if we parsed from content
                if next_message.get("tool_calls") is not None and len(next_message["tool_calls"]) > 0:
                    # Standard tool_calls format
                    next_message["tool_calls"] = next_message["tool_calls"][:1]
                    messages.extend(
                        [
                            next_message,
                            {
                                "role": "tool",
                                "tool_call_id": next_message["tool_calls"][0]["id"],
                                "name": next_message["tool_calls"][0]["function"]["name"],
                                "content": env_response.observation,
                            },
                        ]
                    )
                else:
                    # We parsed function call from content - simulate tool_calls structure
                    fake_tool_call = {
                        "id": f"call_{action.name}_{len(messages)}",
                        "type": "function",
                        "function": {
                            "name": action.name,
                            "arguments": json.dumps(action.kwargs)
                        }
                    }
                    next_message["tool_calls"] = [fake_tool_call]
                    messages.extend(
                        [
                            next_message,
                            {
                                "role": "tool",
                                "tool_call_id": fake_tool_call["id"],
                                "name": action.name,
                                "content": env_response.observation,
                            },
                        ]
                    )
                
                # CRITICAL FIX: After tool execution, agent must process result and respond to user
                log_and_print(f"🔧 DEBUG: Tool executed, now getting agent response to tool result...")
                
                # Make another LLM call for agent to process tool result and respond to user
                completion_params = {
                    "messages": messages,
                    "model": self.litellm_model,
                    "custom_llm_provider": self.litellm_provider,
                    "temperature": self.temperature,
                }
                
                # Only add tools if we have them
                if self.tools_info:
                    completion_params["tools"] = self.tools_info
                    
                try:
                    follow_up_res = completion(**completion_params)
                    log_and_print(f"✅ DEBUG: Follow-up LiteLLM response received")
                    total_cost += follow_up_res._hidden_params["response_cost"]
                    
                    # Extract the follow-up message
                    follow_up_message = follow_up_res.choices[0].message.model_dump()
                    
                    log_and_print(f"🤖 DEBUG: Follow-up message from model:")
                    log_and_print(f"   Role: {follow_up_message.get('role')}")
                    log_and_print(f"   Content: {follow_up_message.get('content')}")
                    log_and_print(f"   Tool calls: {follow_up_message.get('tool_calls')}")
                    
                    # Convert follow-up to action
                    follow_up_action = debug_message_to_action(follow_up_message)
                    
                    log_and_print(f"🎯 DEBUG: Follow-up action created:")
                    log_and_print(f"   Name: {follow_up_action.name}")
                    log_and_print(f"   Kwargs: {follow_up_action.kwargs}")
                    
                    # Execute the follow-up action (should usually be a text response)
                    follow_up_env_response = env.step(follow_up_action)
                    reward = follow_up_env_response.reward
                    info = {**info, **follow_up_env_response.info.model_dump()}
                    
                    log_and_print(f"🔧 DEBUG: Follow-up environment response:")
                    log_and_print(f"   Reward: {reward}")
                    log_and_print(f"   Done: {follow_up_env_response.done}")
                    log_and_print(f"   Observation: {follow_up_env_response.observation}")
                    
                    # Add the follow-up response to conversation
                    if follow_up_action.name != RESPOND_ACTION_NAME:
                        # If somehow agent made another tool call, handle it
                        log_and_print(f"🔧 DEBUG: Adding follow-up tool call to conversation")
                        if follow_up_message.get("tool_calls") is not None and len(follow_up_message["tool_calls"]) > 0:
                            follow_up_message["tool_calls"] = follow_up_message["tool_calls"][:1]
                            messages.extend(
                                [
                                    follow_up_message,
                                    {
                                        "role": "tool",
                                        "tool_call_id": follow_up_message["tool_calls"][0]["id"],
                                        "name": follow_up_message["tool_calls"][0]["function"]["name"],
                                        "content": follow_up_env_response.observation,
                                    },
                                ]
                            )
                        else:
                            fake_follow_up_tool_call = {
                                "id": f"call_{follow_up_action.name}_{len(messages)}",
                                "type": "function",
                                "function": {
                                    "name": follow_up_action.name,
                                    "arguments": json.dumps(follow_up_action.kwargs)
                                }
                            }
                            follow_up_message["tool_calls"] = [fake_follow_up_tool_call]
                            messages.extend(
                                [
                                    follow_up_message,
                                    {
                                        "role": "tool",
                                        "tool_call_id": fake_follow_up_tool_call["id"],
                                        "name": follow_up_action.name,
                                        "content": follow_up_env_response.observation,
                                    },
                                ]
                            )
                    else:
                        # Normal case: agent provided text response after tool execution
                        log_and_print(f"🔧 DEBUG: Adding follow-up text response to conversation")
                        messages.extend(
                            [
                                follow_up_message,
                                {"role": "user", "content": follow_up_env_response.observation},
                            ]
                        )
                    
                    # Update completion status
                    if follow_up_env_response.done:
                        log_and_print(f"🏁 DEBUG: Task completed after follow-up at step {step + 1}")
                        break
                        
                except Exception as e:
                    log_and_print(f"❌ DEBUG: Follow-up LiteLLM call failed: {e}")
                    break
                    
            else:
                log_and_print(f"🔧 DEBUG: Adding text response to conversation")
                messages.extend(
                    [
                        next_message,
                        {"role": "user", "content": env_response.observation},
                    ]
                )
            
            # Log the updated conversation state
            log_and_print(f"\n📝 CONVERSATION STATE AFTER TURN {step + 1}:")
            log_and_print(f"   Total messages: {len(messages)}")
            log_and_print(f"   Current reward: {reward}")
            log_and_print(f"   Task done: {env_response.done}")
            
            if env_response.done:
                log_and_print(f"🏁 DEBUG: Task completed at step {step + 1}")
                break
                
        log_and_print(f"🎯 DEBUG: Final result - Reward: {reward}, Total cost: {total_cost}")
        log_and_print(f"💾 DEBUG: Complete log saved to: {log_filename}")
        
        return SolveResult(
            reward=reward,
            info=info,
            messages=messages,
            total_cost=total_cost,
        )
    


In [ ]:
####### cell #######



import argparse
from dataclasses import dataclass
from tau_bench.types import RunConfig
from tau_bench.run import run
from litellm import provider_list
from tau_bench.envs.user import UserStrategy
import os
import json
import random
import traceback
import time
from math import comb
import multiprocessing
from typing import List, Dict, Any
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

from tau_bench.envs import get_env
from tau_bench.agents.base import Agent
from tau_bench.types import EnvRunResult, RunConfig
from tau_bench.run import display_metrics
from litellm import provider_list
from tau_bench.envs.user import UserStrategy


####### cell #######


# Configuration setup based on the specified parameters
@dataclass
class Config:
    agent_strategy: str = "tool-calling"
    env: str = "retail"
    model: str = "gemma3:12b"
    model_provider: str = "gemma"
    user_model: str = "gemini-2.5-pro"
    user_model_provider: str = "vertex_ai"
    user_strategy: str = "llm"
    max_concurrency: int = 1
    task_id: int = 0
    # Additional config parameters that seem to be expected
    user_temperature: float = 0.0
    task_split: str = "train"
    end_index: int = -1
    start_index: int = 0
    task_ids: List[int] = None
    num_trials: int = 1
    shuffle: bool = False

config = Config()

# Set up task_ids if task_id is specified
if config.task_id is not None:
    config.task_ids = [config.task_id]

# Agent factory function
def agent_factory(tools_info, wiki, config):
    return ToolCallingAgent(
        tools_info=tools_info,
        wiki=wiki,
        model=config.model,
        provider=config.model_provider,
        temperature=0.0
    )


####### cell #######


# Checkpoint path setup
ckpt_path = f"gemma3_debug_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

# Start timing
start_time = time.time()

print(f"Loading user with strategy: {config.user_strategy}")
env = get_env(
    config.env,
    user_strategy=config.user_strategy,
    user_model=config.user_model,
    user_provider=config.user_model_provider,
    user_temperature=config.user_temperature,
    task_split=config.task_split,
)
agent = agent_factory(
    wiki=env.wiki,
    config=config,
    tools_info=env.tools_info,
    #tools_info=None,
)
end_index = (
    len(env.tasks) if config.end_index == -1 else min(config.end_index, len(env.tasks))
)
results: List[EnvRunResult] = []
lock = multiprocessing.Lock()
if config.task_ids and len(config.task_ids) > 0:
    print(f"Running tasks {config.task_ids} (checkpoint path: {ckpt_path})")
else:
    print(
        f"Running tasks {config.start_index} to {end_index} (checkpoint path: {ckpt_path})"
    )
for i in range(config.num_trials):
    if config.task_ids and len(config.task_ids) > 0:
        idxs = config.task_ids
    else:
        idxs = list(range(config.start_index, end_index))
    if config.shuffle:
        random.shuffle(idxs)

    def _run(idx: int) -> EnvRunResult:
        isolated_env = get_env(
            config.env,
            user_strategy=config.user_strategy,
            user_model=config.user_model,
            task_split=config.task_split,
            user_provider=config.user_model_provider,
            user_temperature=config.user_temperature,
            task_index=idx,
        )

        print(f"Running task {idx}")
        try:
            res = agent.solve(
                env=isolated_env,
                task_index=idx,
            )
            result = EnvRunResult(
                task_id=idx,
                reward=res.reward,
                info=res.info,
                traj=res.messages,
                trial=i,
            )
        except Exception as e:
            result = EnvRunResult(
                task_id=idx,
                reward=0.0,
                info={"error": str(e), "traceback": traceback.format_exc()},
                traj=[],
                trial=i,
            )
        print(
            "✅" if result.reward == 1 else "❌",
            f"task_id={idx}",
            result.info,
        )
        print("-----")
        with lock:
            data = []
            if os.path.exists(ckpt_path):
                with open(ckpt_path, "r") as f:
                    data = json.load(f)
            with open(ckpt_path, "w") as f:
                json.dump(data + [result.model_dump()], f, indent=2)
        return result

    with ThreadPoolExecutor(max_workers=config.max_concurrency) as executor:
        res = list(executor.map(_run, idxs))
        results.extend(res)

metrics = display_metrics(results)

# Calculate total runtime
end_time = time.time()
total_runtime = end_time - start_time
runtime_minutes = total_runtime / 60
runtime_hours = total_runtime / 3600

# Print runtime to terminal
print(f"\n⏱️  Total Runtime: {total_runtime:.2f} seconds ({runtime_minutes:.2f} minutes, {runtime_hours:.2f} hours)")

Loading user with strategy: llm
🔧 DEBUG: Agent initialized with:
   Model: gemma3:12b
   Provider: gemma
   Temperature: 0.0
   Wiki length: 5719
   Tools count: 0 -> LIMITED TO: 0
   Available tools: None (no tools provided)
🔧 DEBUG: Mapped to Ollama - gemma3:12b
Running tasks [0] (checkpoint path: gemma3_debug_results_20250918_012254.json)
Running task 0

🚀 DEBUG: Starting solve for task 0
🚀 DEBUG: Starting solve for task 0
📅 Timestamp: 2025-09-18T01:23:07.542014
🔧 Model: gemma3:12b via gemma
🔧 DEBUG: Initial observation: Hello, I would like to start the process for a return.
🔧 DEBUG: Initial messages constructed:
   System message length: 5719
   User message: Hello, I would like to start the process for a return.
   🎯 Task index being run: 0

📋 FULL SYSTEM MESSAGE (WIKI):
# Retail agent policy

As a retail agent, you can help users cancel or modify pending orders, return or exchange delivered orders, modify their default user address, or provide information about their own profile,